# On Reproducibility of LLM-generated Respondents

In this notebook, we use experimental results to verify the reproducibility of LLM-generated respondents.

We use a 50-item questionnaire of the Big Five personality test from **[Open-Source Psychometrics Project](https://openpsychometrics.org/tests/IPIP-BFFM/)** for the experiment. With a set of **prompt**, **random seed**, **large-language-model**, and **temperature setting**, we ask the LLM to respond to the questionnaire multiple times. The generated respondents are then analyzed.

## Experiments

### Questionnaire
[Possible Questionnaire Format for Administering the 50-Item Set of IPIP Big-Five Factor Markers](https://ipip.ori.org/new_ipip-50-item-scale.htm). International Personality Item Pool.

- References: Goldberg, Lewis R. "The development of markers for the Big-Five factor structure." Psychological assessment 4.1 (1992): 26.

### Prompt

1. Basic instruction:
```text
Indicate for the statement whether it is 1. Very Inaccurate, 2. Moderately Inaccurate, 3. Neither Accurate Nor Inaccurate, 4. Moderately Accurate, or 5. Very Accurate as a description of you.
```

2. Format instruction:
```text
Output the results as a json object: {'rating':'1-5' ,'reasoning':'a simple reasoning of the rating in one sentence'}
```

3. Persona:
We can define the LLM's personality and ask it to respond in that persona. For example:
```text
You are a 35-year0old male. You are a senior software engineer at Google. You are married and you have a 2-year-old girl child.
```
The final prompt is: `persona + basic_instruction + item + format_instruction`.

### Random Seed
Most LLM can be used with specified `random_seed` when `temperature` is set to `0.0`.

### The Experiment
We used `gemma4:e4b` to run 100 times with the same settings.


## Analysis

### Load Respondents

In [3]:
## Load CSV data
import os
import pandas as pd

DATAPATH = '../output/default_fixed_all_gemma4e4b/'
files = os.listdir(DATAPATH)
print('Totally '+str(len(files))+' files.')
print('Read the first file: '+DATAPATH+files[0])
df = pd.read_csv(DATAPATH+files[0], index_col=0)
print(df.shape)
print(df.head())

Totally 100 files.
Read the first file: ../output/default_fixed_all_gemma4e4b/ollama-default-exp000.csv
(50, 2)
   rating                                          reasoning
0       2  As a busy senior engineer and father, my energ...
1       2  As a married father with a young child, I am l...
2       2  While I strive to be organized and professiona...
3       2  While work stress is possible, being a senior ...
4       3  My life is characterized by a complex balance ...


In [4]:
## Separating rating and reasoning
ratings = []
reasonings = []
#
for f in files:
    tmp = pd.read_csv(DATAPATH+f, index_col=0)
    ratings.append(tmp['rating'])
    reasonings.append(tmp['reasoning'])

In [8]:
df_ratings= pd.concat(ratings, axis=1).transpose()
df_reasoning = pd.concat(reasonings, axis=1).transpose()

### Verify the Variability of the Responses

In [11]:
df_ratings.std()

0     0.0
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
13    0.0
14    0.0
15    0.0
16    0.0
17    0.0
18    0.0
19    0.0
20    0.0
21    0.0
22    0.0
23    0.0
24    0.0
25    0.0
26    0.0
27    0.0
28    0.0
29    0.0
30    0.0
31    0.0
32    0.0
33    0.0
34    0.0
35    0.0
36    0.0
37    0.0
38    0.0
39    0.0
40    0.0
41    0.0
42    0.0
43    0.0
44    0.0
45    0.0
46    0.0
47    0.0
48    0.0
49    0.0
dtype: float64

As shown above, there is no variability in the 100 tests, and hence the LLM-generated responses is **reproducible with fixed prompt and random-seed**. Note that we only checked a few LLMs, and the conclusion may not generalize to all LLMs.


## References:
- [Assessing the Reliability of Persona-Conditioned LLMs as Synthetic Survey Respondents](https://arxiv.org/abs/2602.18462v1)
- [Polypersona: Persona-Grounded LLM for Synthetic Survey Responses](https://arxiv.org/abs/2512.14562v1)
- [Persona Generators: Generating Diverse Synthetic Personas at Scale](https://arxiv.org/abs/2602.03545v1)